# Optimizing Agentic AI Collaboration v2.2.3 — Qwen3 Thinking-Mode GSM8K
## Sequential Evaluation of Qwen3-4B and Qwen3-1.7B with Chain-of-Thought

**Models:** `Qwen/Qwen3-4B` and `Qwen/Qwen3-1.7B` with 4-bit NF4 quantization, evaluated sequentially.
**Dataset:** GSM8K (128 problems × 3 seeds per model).
**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier).

## What this notebook is

This notebook evaluates **Qwen3-4B** and **Qwen3-1.7B** on **GSM8K** in **thinking mode** (`/think` soft switch, `max_new_tokens=2048`, `temp=0.6`) — a complement to the v2.2.x non-thinking baselines that quantifies the accuracy and cost impact of Qwen3's chain-of-thought reasoning trace. The thinking-mode boost is expected to be largest for the smallest model (Qwen3-1.7B), where the non-thinking baseline scores lowest. **Qwen3-8B is intentionally excluded**: its v2.2.0 non-thinking baseline already scores above 0.9 accuracy on GSM8K while decoding slower than the smaller models, so spending the thinking-mode compute budget on it would buy little to no additional accuracy.

Both models are loaded, evaluated, and unloaded **sequentially** in a single session to fit within the 16 GB T4 VRAM budget. A dedicated **W&B run** is created per model for direct comparability. Each of the **128 GSM8K problems** is evaluated under **3 generation seeds** (42 / 123 / 456).

## Relation with the project WPs

**WP2 - T3: Single-agent baseline establishment (CoT).** This notebook establishes 2 zero-shot, CoT, single-agent baselines, for the reasnoning mode of 2 of the discuted state-of-the-art SLMs (Qwen-3-1.7B and Qwen-3-4B) over GSM8K. Theses baseline are the references the L1/L2 multi-agent protocols (WP2 - T4/T6) must overcome, and their latency/token/VRAM measurements feed the WP3 - T1 compute-accuracy trade-off analysis.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to `cuda:0`).
2. Add the following secrets via **Add-ons → Secrets**, enabling each for the notebook:
   - `GITHUB_TOKEN` — GitHub PAT with repo read scope
   - `HF_TOKEN` — Hugging Face read-access token
   - `WANDB_API_KEY` — Weights & Biases API key
3. Click **Run All**. The first cell auto-detects Kaggle, clones the repo, and loads secrets.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token
   - `WANDB_API_KEY` — your Weights & Biases API key
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets

## Work package

**WP2 · T3 — Establishment of robust single-agent baselines.** This notebook establishes a zero-shot / Chain-of-Thought single-agent baseline for one SLM over GSM8K and/or MMLU. These baselines are the reference the L1/L2 multi-agent protocols (WP2 · T4/T6) must beat by ≥5%, and their latency/token/VRAM measurements feed the WP3 · T1 trade-off analysis.


## What this notebook implements

A **thinking-mode GSM8K baseline** over two Qwen3 model sizes with W&B tracking:

* A single direct-inference call per problem (no multi-agent pipeline), with Qwen3's chain-of-thought trace enabled via the `/think` soft switch (`max_new_tokens=2048`).
* **W&B logging** of per-call latency, output/total token counts, GPU peak VRAM, and generation throughput; plus a unified W&B `Table` of every sample.
* Each problem is run under **3 generation seeds**; correctness, latency, token and peak-VRAM matrices (N×m) are persisted to a W&B artifact for downstream analysis.
* **Qwen3-4B** is evaluated first, then the model is unloaded and **Qwen3-1.7B** is loaded and evaluated — both in the same session.

Each model produces one W&B run for direct cross-model comparability. Overhead ratios/deltas are **not** computed here.

## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `datasets`, `wandb`), all pinned in `requirements.txt`. If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [ ]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

In [ ]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [ ]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

## 3. Shared Configuration

`GEN` and `CONFIG_BASE` are shared across both model evaluations. `model_id` and `run_name` are defined per-model in each model's load cell.

In [ ]:
PROJECT = "agentic-final-results" if (ENVIRONMENT == "kaggle") else "dev-agentic-baselines"
print(f"W&B Project: {PROJECT}")

# Generation config — thinking mode for both models.
GEN = {
    "max_new_tokens": 2048,
    "temperature":    0.6,
    "top_p":          0.95,
    "top_k":          20,
}

# Shared config keys (model_id and run_name are set per-model in their load cell).
CONFIG_BASE = {
    "quantization":     "nf4-4bit",
    "framework":        "langchain",
    "protocol":         "single_agent_baseline",
    "eval_size":        128,
    "data_seed":        789,
    "generation_seeds": [42, 123, 456],
    "thinking_mode":    True,
}

## 4. Data Loading

We load GSM8K upfront and reuse it for both model evaluations.

In [ ]:
from utils.data import load_gsm8k

gsm8k_dataset = load_gsm8k(split="test")

print(f"✅ GSM8K dataset loaded with {len(gsm8k_dataset)} examples.")
print(f"Example question: {gsm8k_dataset[0]['question']}")
print(f"Example answer:  {gsm8k_dataset[0]['answer']}")

## 5. Thinking-Mode Template

The `/think` soft switch prepended to the user message enables Qwen3's internal chain-of-thought trace. Both `run_agent_step` and `run_single_agent_baseline` reference `gsm8k_template` and `tokenizer` as globals; rebinding `tokenizer` before each model section is sufficient — the functions use the new binding automatically.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# /think enables Qwen3's chain-of-thought trace for this turn.
# The thinking trace appears before \boxed{} in the output; extract_gsm8k_prediction
# scans for the LAST \boxed{} so it is robust to reasoning traces.
gsm8k_template = PromptTemplate.from_template(
    "/think\n\n"
    "{task}\n\n"
    "Please reason step by step, and put your final answer within \\boxed{{}}."
)

print("✅ Thinking-mode GSM8K template defined.")

## 6. Single-Agent Runner with W&B Logging

`run_agent_step` times a single `chain.invoke`, counts tokens outside the timed region, and measures per-call peak VRAM. `run_single_agent_baseline` wraps one baseline call and logs latency, tokens, peak VRAM and throughput per sample. Both functions are defined once and reused for both model sections.

In [ ]:
import time
from utils.logging_utils import log_metrics
from utils.instrumentation import (
    count_tokens, render_prompt,
    reset_gpu_peak, gpu_peak_mem_mb,
    tokens_per_second,
)

def run_agent_step(chain, inputs, step_name, prompt_template):
    """Run a single LangChain chain, time it, count tokens, measure per-step peak VRAM and throughput.

    Token counting and VRAM measurement are performed outside the timed region
    so the latency measurement wraps only ``chain.invoke``.

    Returns
    -------
    tuple[str, float, int, int, float]
        (output_text, elapsed_s, input_tokens, output_tokens, peak_vram_mb)
    """
    prompt_str   = render_prompt(prompt_template, inputs)
    input_tokens = count_tokens(tokenizer, prompt_str)

    reset_gpu_peak()

    t0          = time.perf_counter()
    output_text = chain.invoke(inputs)
    elapsed     = time.perf_counter() - t0

    output_tokens = count_tokens(tokenizer, output_text)
    total_tokens  = input_tokens + output_tokens
    peak_vram_mb  = gpu_peak_mem_mb()
    tps           = tokens_per_second(output_tokens, elapsed)

    print(
        f"  ⏱  {step_name}: {elapsed:.1f}s  |  "
        f"{len(output_text)} chars  |  "
        f"in={input_tokens} out={output_tokens} tok  |  "
        f"peak={peak_vram_mb:.0f} MB  |  {tps:.1f} tok/s"
    )
    return output_text, elapsed, input_tokens, output_tokens, peak_vram_mb


def run_single_agent_baseline(chain, task, task_label="task"):
    """Execute a single-agent baseline call with W&B logging.

    Returns
    -------
    dict with keys: baseline, t_base, base_in_tok, base_out_tok,
                    base_total_tok, base_peak_mb, base_throughput
    """
    print(f"\n{'='*60}")
    print(f"📊 SINGLE-AGENT BASELINE  —  {task_label}")
    print(f"{'='*60}")

    baseline, t_base, in_base, out_base, peak_base = run_agent_step(
        chain, {"task": task},
        f"{task_label}/baseline", gsm8k_template
    )

    base_total_tok  = in_base + out_base
    base_throughput = tokens_per_second(out_base, t_base)

    log_metrics({
        f"{task_label}/latency_s":      round(t_base, 2),
        f"{task_label}/output_tokens":  out_base,
        f"{task_label}/total_tokens":   base_total_tok,
        f"{task_label}/gpu_peak_mb":    round(peak_base, 2),
        f"{task_label}/throughput_tps": round(base_throughput, 2),
    })

    return {
        "baseline":        baseline,
        "t_base":          t_base,
        "base_in_tok":     in_base,
        "base_out_tok":    out_base,
        "base_total_tok":  base_total_tok,
        "base_peak_mb":    peak_base,
        "base_throughput": base_throughput,
    }

print("✅ Single-agent runner ready.")

---
## 7. Qwen3-4B — Model Loading (4-bit Quantized)

Qwen3-4B (~3 GB NF4) is the first model loaded in this session, via the standard `AutoModelForCausalLM` path quantized to NF4 4-bit (`load_model_and_tokenizer`).

In [ ]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = "Qwen/Qwen3-4B"
run_name = "GSM8K-Thinking-Qwen3-4B"

model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)
print(f"✅ {model_id} loaded on {model.device} — dtype {model.dtype}")

## 8. Qwen3-4B — Warmup

In [ ]:
from utils.warmup import warmup_model

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=GEN["temperature"],
    top_p=GEN["top_p"], top_k=GEN["top_k"],
)

## 9. Qwen3-4B — Initialize W&B

In [ ]:
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics

run = init_wandb(
    run_name=run_name,
    project=PROJECT,
    config={**CONFIG_BASE, **GEN, "model_id": model_id, "eval_dataset": "gsm8k"}
)
print(f"✅ W&B run: {run.name}  —  {run.url}")

define_wandb_chart_only_metrics()

## 10. Qwen3-4B — GSM8K Evaluation

We evaluate the single-agent baseline over the fixed 128-problem subset, repeating each problem under 3 generation seeds. The `run_agent_step` and `run_single_agent_baseline` functions defined in section 6 operate over the `model` and `tokenizer` bound in section 7 (Qwen3-4B).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
from tqdm.notebook import tqdm
from utils.logging_utils import log_metrics
from utils.agents import build_hf_langchain_llm
from utils.data import extract_gsm8k_answer
from utils.evaluation import extract_gsm8k_prediction, gsm8k_match
from langchain_core.output_parsers import StrOutputParser

generation_seeds = wandb.config.generation_seeds
sample_size      = wandb.config.eval_size

random.seed(wandb.config.data_seed)
test_sample = random.sample(list(gsm8k_dataset), sample_size)

baseline_correctness_matrix = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_time_matrix        = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_input_tok_matrix   = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_output_tok_matrix  = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_gpu_peak_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=float)

results_columns = [
    "model_id", "dataset", "protocol",
    "problem_id", "seed",
    "task",
    "output",
    "ground_truth", "final_output",
    "latency_s",
    "output_tokens", "total_tokens",
    "gpu_peak_mb",
    "throughput_tok_s",
]
results_rows = []

for run_idx, gen_seed in enumerate(generation_seeds):
    print(f"\n{'='*60}")
    print(f"  Seed run {run_idx + 1}/{len(generation_seeds)}  —  generation_seed={gen_seed}")
    print(f"{'='*60}")

    llm = build_hf_langchain_llm(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=wandb.config.max_new_tokens,
        temperature=wandb.config.temperature,
        top_k=wandb.config.top_k,
        top_p=wandb.config.top_p,
        return_full_text=False,
        seed=gen_seed,
    )
    parser = StrOutputParser()
    baseline_chain = gsm8k_template | llm | parser

    for item_idx, item in enumerate(tqdm(test_sample, desc=f"GSM8K seed={gen_seed} ({sample_size} samples)")):
        task              = item['question']
        ground_truth_full = item['answer']
        true_answer       = extract_gsm8k_answer(ground_truth_full)

        output = None
        try:
            output            = run_single_agent_baseline(baseline_chain, task, task_label='gsm8k_sample')
            baseline_response = output['baseline']
            baseline_answer   = extract_gsm8k_prediction(baseline_response)
        except Exception as e:
            print(f"Baseline failed: {e}")
            baseline_answer   = None
            baseline_response = str(e)
            if output is None:
                output = {
                    'baseline': baseline_response,
                    't_base': 0.0,
                    'base_in_tok': 0, 'base_out_tok': 0, 'base_total_tok': 0,
                    'base_peak_mb': 0.0, 'base_throughput': 0.0,
                }

        baseline_correct = gsm8k_match(baseline_answer, true_answer)

        baseline_correctness_matrix[item_idx, run_idx] = int(baseline_correct)
        baseline_time_matrix[item_idx, run_idx]        = output['t_base']
        baseline_input_tok_matrix[item_idx, run_idx]   = output['base_in_tok']
        baseline_output_tok_matrix[item_idx, run_idx]  = output['base_out_tok']
        baseline_gpu_peak_matrix[item_idx, run_idx]    = output['base_peak_mb']

        log_metrics({"gsm8k_sample/accuracy": int(baseline_correct)})

        results_rows.append([
            model_id, "gsm8k", "single_agent_baseline",
            item_idx, gen_seed,
            task,
            output['baseline'][:2000],
            true_answer, baseline_answer,
            round(output['t_base'], 2),
            output['base_out_tok'], output['base_total_tok'],
            round(output['base_peak_mb'], 2),
            round(output['base_throughput'], 4),
        ])

results_rows.sort(key=lambda r: (r[3], r[4]))
results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({"results_comparison_GSM8K": results_table})

## 11. Qwen3-4B — Results, Statistics & Finish W&B Run

In [ ]:
import tempfile
import wandb
import numpy as np
from utils.logging_utils import finish_wandb, log_stat_summary
from utils.statistics import (
    accuracy_statistics,
    time_statistics,
    token_statistics,
    vram_statistics,
    print_accuracy_stats_report,
    print_time_report,
    print_token_report,
    print_vram_report,
)

DATASET = "GSM8K"

baseline_total_tok_matrix = baseline_input_tok_matrix + baseline_output_tok_matrix

baseline_acc_stats      = accuracy_statistics(baseline_correctness_matrix)
baseline_time_stats     = time_statistics(baseline_time_matrix)
baseline_out_tok_stats  = token_statistics(baseline_output_tok_matrix, max_tokens=wandb.config.max_new_tokens)
baseline_total_tok_stats = token_statistics(baseline_total_tok_matrix)
baseline_vram_stats     = vram_statistics(baseline_gpu_peak_matrix)

print_accuracy_stats_report(f"Baseline Accuracy ({DATASET})", baseline_acc_stats)
print_time_report(f"Baseline Latency ({DATASET})", baseline_time_stats)
print_token_report(f"Baseline Output Tokens ({DATASET})", baseline_out_tok_stats)
print_token_report(f"Baseline Total Tokens ({DATASET})", baseline_total_tok_stats)
print_vram_report(f"Baseline Peak VRAM ({DATASET})", baseline_vram_stats)

wandb.summary["n_seeds"]           = len(generation_seeds)
wandb.summary["n_problems"]        = sample_size
wandb.summary["total_experiments"] = sample_size * len(generation_seeds)

wandb.summary["accuracy"] = round(baseline_acc_stats["p_hat"], 6)
log_stat_summary("accuracy_stats", baseline_acc_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "wilson_ci_lower", "wilson_ci_upper", "wilson_n_eff", "wilson_icc"])

wandb.summary["latency"] = round(baseline_time_stats["mean"], 2)
log_stat_summary("latency_stats", baseline_time_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "flat_cv"])

wandb.summary["output_tokens"] = round(baseline_out_tok_stats["mean"], 1)
log_stat_summary("output_tokens_stats", baseline_out_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index", "frac_at_ci_lower", "frac_at_ci_upper"])

wandb.summary["total_tokens"] = round(baseline_total_tok_stats["mean"], 1)
log_stat_summary("total_tokens_stats", baseline_total_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])

wandb.summary["mean_vram_peak"] = round(baseline_vram_stats["mean_peak"], 0)
wandb.summary["max_vram_peak"]  = round(baseline_vram_stats["peak_max"], 0)
log_stat_summary("vram_peak_stats", baseline_vram_stats,
    keys=["peak_ci_lower", "peak_ci_upper", "between_problem_peak_std"])

artifact = wandb.Artifact(
    "evaluation-matrices-Qwen-3-4B-Thinking-GSM8K",
    type="evaluation-data",
    description=(
        f"Thinking-mode single-agent baseline evaluation matrices ({len(generation_seeds)} x {sample_size}). "
        f"GSM8K evaluation with Qwen/Qwen3-4B. "
        "Includes: correctness, execution times, input/output/total token counts, GPU peak VRAM (MB)."
    ),
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        baseline_correctness=baseline_correctness_matrix,
        baseline_times=baseline_time_matrix,
        baseline_input_tokens=baseline_input_tok_matrix,
        baseline_output_tokens=baseline_output_tok_matrix,
        baseline_total_tokens=baseline_total_tok_matrix,
        baseline_gpu_peak=baseline_gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)
print("✅ All evaluation matrices uploaded as W&B artifact.")

finish_wandb()
print("W&B run finished.")

## 12. Qwen3-4B — Cleanup

We delete the model and tokenizer and free GPU VRAM before loading Qwen3-1.7B.

In [ ]:
import gc
import torch

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"✅ {model_id} unloaded. VRAM freed for Qwen3-1.7B.")

---
## 13. Qwen3-1.7B — Model Loading (4-bit Quantized)

Qwen3-1.7B (~1.5 GB NF4) loads into the VRAM freed by the Qwen3-4B cleanup. Same `AutoModelForCausalLM` path and tokenizer as Qwen3-4B.

In [ ]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = "Qwen/Qwen3-1.7B"
run_name = "GSM8K-Thinking-Qwen3-1.7B"

model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)
print(f"✅ {model_id} loaded on {model.device} — dtype {model.dtype}")

## 14. Qwen3-1.7B — Warmup

In [ ]:
from utils.warmup import warmup_model

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=GEN["temperature"],
    top_p=GEN["top_p"], top_k=GEN["top_k"],
)

## 15. Qwen3-1.7B — Initialize W&B

In [ ]:
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics

run = init_wandb(
    run_name=run_name,
    project=PROJECT,
    config={**CONFIG_BASE, **GEN, "model_id": model_id, "eval_dataset": "gsm8k"}
)
print(f"✅ W&B run: {run.name}  —  {run.url}")

define_wandb_chart_only_metrics()

## 16. Qwen3-1.7B — GSM8K Evaluation

Same evaluation loop as Qwen3-4B. The `run_agent_step` and `run_single_agent_baseline` functions now operate over the newly bound `model` and `tokenizer` (Qwen3-1.7B).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
from tqdm.notebook import tqdm
from utils.logging_utils import log_metrics
from utils.agents import build_hf_langchain_llm
from utils.data import extract_gsm8k_answer
from utils.evaluation import extract_gsm8k_prediction, gsm8k_match
from langchain_core.output_parsers import StrOutputParser

generation_seeds = wandb.config.generation_seeds
sample_size      = wandb.config.eval_size

random.seed(wandb.config.data_seed)
test_sample = random.sample(list(gsm8k_dataset), sample_size)

baseline_correctness_matrix = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_time_matrix        = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_input_tok_matrix   = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_output_tok_matrix  = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_gpu_peak_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=float)

results_columns = [
    "model_id", "dataset", "protocol",
    "problem_id", "seed",
    "task",
    "output",
    "ground_truth", "final_output",
    "latency_s",
    "output_tokens", "total_tokens",
    "gpu_peak_mb",
    "throughput_tok_s",
]
results_rows = []

for run_idx, gen_seed in enumerate(generation_seeds):
    print(f"\n{'='*60}")
    print(f"  Seed run {run_idx + 1}/{len(generation_seeds)}  —  generation_seed={gen_seed}")
    print(f"{'='*60}")

    llm = build_hf_langchain_llm(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=wandb.config.max_new_tokens,
        temperature=wandb.config.temperature,
        top_k=wandb.config.top_k,
        top_p=wandb.config.top_p,
        return_full_text=False,
        seed=gen_seed,
    )
    parser = StrOutputParser()
    baseline_chain = gsm8k_template | llm | parser

    for item_idx, item in enumerate(tqdm(test_sample, desc=f"GSM8K seed={gen_seed} ({sample_size} samples)")):
        task              = item['question']
        ground_truth_full = item['answer']
        true_answer       = extract_gsm8k_answer(ground_truth_full)

        output = None
        try:
            output            = run_single_agent_baseline(baseline_chain, task, task_label='gsm8k_sample')
            baseline_response = output['baseline']
            baseline_answer   = extract_gsm8k_prediction(baseline_response)
        except Exception as e:
            print(f"Baseline failed: {e}")
            baseline_answer   = None
            baseline_response = str(e)
            if output is None:
                output = {
                    'baseline': baseline_response,
                    't_base': 0.0,
                    'base_in_tok': 0, 'base_out_tok': 0, 'base_total_tok': 0,
                    'base_peak_mb': 0.0, 'base_throughput': 0.0,
                }

        baseline_correct = gsm8k_match(baseline_answer, true_answer)

        baseline_correctness_matrix[item_idx, run_idx] = int(baseline_correct)
        baseline_time_matrix[item_idx, run_idx]        = output['t_base']
        baseline_input_tok_matrix[item_idx, run_idx]   = output['base_in_tok']
        baseline_output_tok_matrix[item_idx, run_idx]  = output['base_out_tok']
        baseline_gpu_peak_matrix[item_idx, run_idx]    = output['base_peak_mb']

        log_metrics({"gsm8k_sample/accuracy": int(baseline_correct)})

        results_rows.append([
            model_id, "gsm8k", "single_agent_baseline",
            item_idx, gen_seed,
            task,
            output['baseline'][:2000],
            true_answer, baseline_answer,
            round(output['t_base'], 2),
            output['base_out_tok'], output['base_total_tok'],
            round(output['base_peak_mb'], 2),
            round(output['base_throughput'], 4),
        ])

results_rows.sort(key=lambda r: (r[3], r[4]))
results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({"results_comparison_GSM8K": results_table})

## 17. Qwen3-1.7B — Results, Statistics & Finish W&B Run

In [ ]:
import tempfile
import wandb
import numpy as np
from utils.logging_utils import finish_wandb, log_stat_summary
from utils.statistics import (
    accuracy_statistics,
    time_statistics,
    token_statistics,
    vram_statistics,
    print_accuracy_stats_report,
    print_time_report,
    print_token_report,
    print_vram_report,
)

DATASET = "GSM8K"

baseline_total_tok_matrix = baseline_input_tok_matrix + baseline_output_tok_matrix

baseline_acc_stats      = accuracy_statistics(baseline_correctness_matrix)
baseline_time_stats     = time_statistics(baseline_time_matrix)
baseline_out_tok_stats  = token_statistics(baseline_output_tok_matrix, max_tokens=wandb.config.max_new_tokens)
baseline_total_tok_stats = token_statistics(baseline_total_tok_matrix)
baseline_vram_stats     = vram_statistics(baseline_gpu_peak_matrix)

print_accuracy_stats_report(f"Baseline Accuracy ({DATASET})", baseline_acc_stats)
print_time_report(f"Baseline Latency ({DATASET})", baseline_time_stats)
print_token_report(f"Baseline Output Tokens ({DATASET})", baseline_out_tok_stats)
print_token_report(f"Baseline Total Tokens ({DATASET})", baseline_total_tok_stats)
print_vram_report(f"Baseline Peak VRAM ({DATASET})", baseline_vram_stats)

wandb.summary["n_seeds"]           = len(generation_seeds)
wandb.summary["n_problems"]        = sample_size
wandb.summary["total_experiments"] = sample_size * len(generation_seeds)

wandb.summary["accuracy"] = round(baseline_acc_stats["p_hat"], 6)
log_stat_summary("accuracy_stats", baseline_acc_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "wilson_ci_lower", "wilson_ci_upper", "wilson_n_eff", "wilson_icc"])

wandb.summary["latency"] = round(baseline_time_stats["mean"], 2)
log_stat_summary("latency_stats", baseline_time_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "flat_cv"])

wandb.summary["output_tokens"] = round(baseline_out_tok_stats["mean"], 1)
log_stat_summary("output_tokens_stats", baseline_out_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index", "frac_at_ci_lower", "frac_at_ci_upper"])

wandb.summary["total_tokens"] = round(baseline_total_tok_stats["mean"], 1)
log_stat_summary("total_tokens_stats", baseline_total_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])

wandb.summary["mean_vram_peak"] = round(baseline_vram_stats["mean_peak"], 0)
wandb.summary["max_vram_peak"]  = round(baseline_vram_stats["peak_max"], 0)
log_stat_summary("vram_peak_stats", baseline_vram_stats,
    keys=["peak_ci_lower", "peak_ci_upper", "between_problem_peak_std"])

artifact = wandb.Artifact(
    "evaluation-matrices-Qwen-3-1.7B-Thinking-GSM8K",
    type="evaluation-data",
    description=(
        f"Thinking-mode single-agent baseline evaluation matrices ({len(generation_seeds)} x {sample_size}). "
        f"GSM8K evaluation with Qwen/Qwen3-1.7B. "
        "Includes: correctness, execution times, input/output/total token counts, GPU peak VRAM (MB)."
    ),
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        baseline_correctness=baseline_correctness_matrix,
        baseline_times=baseline_time_matrix,
        baseline_input_tokens=baseline_input_tok_matrix,
        baseline_output_tokens=baseline_output_tok_matrix,
        baseline_total_tokens=baseline_total_tok_matrix,
        baseline_gpu_peak=baseline_gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)
print("✅ All evaluation matrices uploaded as W&B artifact.")

finish_wandb()
print("W&B run finished.")

## 18. Analysis & Next Steps

### What W&B gives you (per model run)
- **Correctness / accuracy** — per-sample and aggregate baseline accuracy with a 95% CI.
- **Latency, token and VRAM panels** — per-sample `gsm8k_sample/baseline_*` metrics and their cluster-robust summaries.
- **Results table** — every sample's prompt, output (reasoning trace + answer), predicted vs. ground-truth answer, latency, tokens, peak VRAM and throughput.
- **Artifact** — `evaluation_matrices.npz` with the baseline correctness / latency / token / VRAM matrices (N×m).

### Role of this notebook
These runs provide the **thinking-mode GSM8K baseline** (`/think`, 2048-token cap) for Qwen3-4B and Qwen3-1.7B. Paired with the non-thinking results from v2.2.1 and v2.2.2, they quantify the accuracy and cost trade-off of enabling Qwen3's chain-of-thought trace for the two smaller, more efficient model sizes — the effect is expected to be largest for the 1.7B model. Qwen3-8B is intentionally excluded: its v2.2.0 non-thinking baseline already exceeds 0.9 accuracy on GSM8K at a lower decode throughput than the smaller models, so the thinking-mode compute cost would buy little to no additional accuracy. **Overhead ratios/deltas vs. MAS are not computed here** — the MAS notebooks will load these artifacts for comparison.

### Next steps
1. Build the multi-agent system (Society of Minds) from instances of these models and compare accuracy and cost against the non-thinking baselines (OBJ-1,2,3).